In [ ]:
import pandas as pd
import numpy as np
from src.evaluate import evaluate_model
from src.data_engineering import add_trend, add_house_age, add_renovation_flag

In [ ]:
df = pd.read_csv('data/kc_house_data.csv')
unseen_data = pd.read_csv('data/future_unseen_examples.csv')
demographics = pd.read_csv('data/zipcode_demographics.csv')

In [ ]:
df

In [ ]:
unseen_data = unseen_data.merge(demographics, on='zipcode', how='left')

In [ ]:
current_model = pd.read_pickle('model/champion_model.pkl')

In [ ]:
import json
model_features = json.load(open('model/champion_model_features.json', 'r'))

def predict_price(input_df, model, model_features):
    input_df = input_df[model_features]
    prediction = model.predict(input_df)
    return prediction

In [ ]:
predict_price(unseen_data, current_model, model_features)
#458520

In [ ]:
unseen_data.columns

In [ ]:
#df = df.merge(demographics, on='zipcode', how='left')

In [ ]:
df.isnull().sum()

In [ ]:
df.iloc[:, 2:].hist(figsize=(20,20), xrot=-45)

In [ ]:
df.bathrooms.round(0).value_counts()

In [ ]:
df.bedrooms.value_counts()

In [ ]:
df.loc[df.bedrooms>6, 'bedrooms'] = 6
df.loc[df.bathrooms>4, 'bedrooms'] = 4

df["bathrooms"] = df["bathrooms"].round(0).astype(int)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

mask = np.zeros_like(df.iloc[:, 2:].corr(), dtype=bool)
mask[np.triu_indices_from(mask)] = True
# heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(df.iloc[:, 2:].corr()*100, 
           cmap='RdBu_r', 
           annot = True, 
           mask = mask)

In [ ]:
df.iloc[df.yr_built.idxmax()]

In [ ]:
df = df.merge(demographics[["zipcode", "hous_val_amt"]], on='zipcode', how='left')
cols_to_drop = ["zipcode", "sqft_living15", "sqft_lot15"]
data = df.drop(columns=cols_to_drop)

In [ ]:
data["date"] = pd.to_datetime(data.date)

In [ ]:
data.date.max()

In [ ]:
data.date.max() - data.date.min()

In [ ]:
data = add_trend(data)
data = add_renovation_flag(data)
data = add_house_age(data)
data = add_house_age(data, built_col='yr_renovated', age_col='renovation_age')

In [ ]:
from datetime import timedelta

last_date = data.date.max()
val_data = data[data.date > (last_date - timedelta(30))]

In [ ]:
data.drop(val_data.index, inplace=True)
last_date = data.date.max()
test_data = data[data.date > (last_date - timedelta(30))]
data.drop(test_data.index, inplace=True)

In [ ]:
X_train = data.drop(columns=['id', 'date', 'price'])
y_train = data['price']

X_test = test_data.drop(columns=['id', 'date', 'price'])
y_test = test_data['price']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# 1) Define the search space
space = {
    'n_estimators':    hp.choice('n_estimators',    [100, 200, 300, 500]),
    'max_depth':       hp.quniform('max_depth',    3, 10, 1),
    'learning_rate':   hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'subsample':       hp.uniform('subsample',      0.6, 1.0),
    'colsample_bytree':hp.uniform('colsample_bytree',0.6, 1.0),
    'gamma':           hp.uniform('gamma',          0,   5),
    'min_child_weight':hp.quniform('min_child_weight', 1, 10, 1),
    'reg_alpha':       hp.loguniform('reg_alpha',    np.log(1e-3), np.log(10)),
    'reg_lambda':      hp.loguniform('reg_lambda',   np.log(1e-3), np.log(10)),
}

# 2) Objective: 5‑fold CV MAPE
def objective(params):
    # cast integers
    params['max_depth'] = int(params['max_depth'])
    params['min_child_weight'] = int(params['min_child_weight'])

    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        **params
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    neg_mape = cross_val_score(
        model,
        X_train, y_train,
        scoring='neg_mean_absolute_percentage_error',
        cv=cv,
        n_jobs=-1
    ).mean()
    mape = -neg_mape
    return {'loss': mape, 'status': STATUS_OK}

# 3) Run optimization
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=30,
    trials=trials,
    rstate=np.random.default_rng(42)
)

# Convert choice indices back to values
best['n_estimators']     = [100, 200, 300, 500][best['n_estimators']]
best['max_depth']        = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Best hyperparameters:", best)

# 4) Train best model on full training set
best_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    **best
)
best_model.fit(X_train, y_train)

# 5) Evaluate on the hold‑out
y_pred = best_model.predict(X_test)
test_mse  = np.mean((y_test - y_pred)**2)
test_mae  = np.mean(np.abs(y_test - y_pred))
test_mape = np.mean(np.abs((y_test - y_pred) / y_test))
print({
    'MSE':  test_mse,
    'MAE':  test_mae,
    'MAPE': test_mape
})

importances = best_model.feature_importances_
idx = np.argsort(importances)[::-1]
feats = X_train.columns[idx]
imps = importances[idx]

plt.figure(figsize=(12, 8))
plt.barh(feats, imps)
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("XGBRegressor Feature Importances (desc.)")
plt.tight_layout()
plt.show()

#base line = {'MSE': 13550209029.73716, 'MAE': 66886.68607613909, 'MAPE': 0.11862540179763019}


In [ ]:
X_val = val_data.drop(columns=['id', 'date', 'price'])
y_val = val_data['price']

In [ ]:
evaluate_model(best_model,val_data.drop(columns=['id', 'date']),model_features=X_val.columns)


In [ ]:
metrics = evaluate_model(
    current_model,
    test_data,
    full_df=df,                 # the original DataFrame before any splitting
    demographics=demographics,  # zipcode_demographics DataFrame
    model_features=model_features # the exact cols KNN expects
)
print(metrics)

json.dump(metrics, open('model/champion_model_metrics.json', 'w'))

In [ ]:
metrics = evaluate_model(
    current_model,
    val_data,
    full_df=df,                 # the original DataFrame before any splitting
    demographics=demographics,  # zipcode_demographics DataFrame
    model_features=model_features # the exact cols KNN expects
)
print(metrics)


In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import os

merged_data =  df.drop(columns=["hous_val_amt"]).iloc[X_test.index].merge(demographics, on='zipcode', how='left')[model_features]
result_plot = permutation_importance(current_model, merged_data, y_test, n_repeats=10, random_state=42)
importances_plot = result_plot.importances_mean
sorted_idx = np.argsort(importances_plot)
plt.figure(figsize=(10, 8))
plt.barh(merged_data.columns[sorted_idx], importances_plot[sorted_idx])
plt.xlabel("Permutation Importance")
plt.title("Feature Permutation Importance")

fi_path = os.path.join("model", "champion_model_feature_importance.png")
plt.savefig(fi_path)
plt.close()
